# Real-world standalone translator acceptance

This notebook is a qualitative acceptance test for the standalone-translator refactor.
It deliberately uses the frozen **Wikinews March 2025** real-text fixture already in TeAL rather than synthetic strings or a network download.

The goals are to inspect whether the architecture behaves sensibly on messy real text, not to duplicate pytest exhaustively. We will:

1. inspect the real corpus and hold out articles that TeAL never sees during fitting;
2. fit a Count → FeatureTrimmer → TF-IDF → SVD representation inside TeAL;
3. close the project, reload the frozen translators, close the project again, and translate held-out articles using ordinary Python objects only;
4. compare that manual standalone chain against GeCo's private frozen-representation replay;
5. inspect vocabulary, sparsity, held-out coordinates, and top weighted terms qualitatively;
6. exercise cardinality-changing and structured translators on real article paragraphs;
7. optionally run spaCy on the held-out text if `en_core_web_sm` is already installed.

The fixture is frozen, attributed to Wikinews, and licensed CC BY 4.0. No network access is required.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse

import text_analysis_lab as teal
from text_analysis_lab.integrations._representation_replay import _replay_texts_like
from text_analysis_lab.translators import (
    CountVectorizer,
    DelimiterDecomposer,
    FeatureTrimmer,
    RegexCleaner,
    RegexReplaceRule,
    SVD,
    SpacyTranslator,
    TextLength,
    TfidfTransformer,
)


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        fixture = candidate / 'tests' / 'fixtures' / 'wikinews_2025_small' / 'articles.jsonl'
        if (candidate / 'pyproject.toml').exists() and fixture.exists():
            return candidate
    raise RuntimeError('Run this notebook from somewhere inside the TeAL repository.')


REPO_ROOT = find_repo_root()
FIXTURE = REPO_ROOT / 'tests' / 'fixtures' / 'wikinews_2025_small' / 'articles.jsonl'
WORKDIR = Path(tempfile.mkdtemp(prefix='teal-real-world-acceptance-'))
PROJECT_PATH = WORKDIR / 'project'

print('Repository:', REPO_ROOT)
print('Working directory:', WORKDIR)


In [ ]:
rows = [json.loads(line) for line in FIXTURE.read_text(encoding='utf-8').splitlines() if line.strip()]
articles = pd.DataFrame(rows).sort_values('article_id').reset_index(drop=True)

display(articles[['article_id', 'title', 'topic_group', 'region_group', 'published_date']])

corpus_audit = pd.DataFrame({
    'n_articles': [len(articles)],
    'topics': [articles['topic_group'].nunique()],
    'regions': [articles['region_group'].nunique()],
    'min_chars': [articles['text'].str.len().min()],
    'median_chars': [articles['text'].str.len().median()],
    'max_chars': [articles['text'].str.len().max()],
    'paragraphs': [articles['text'].str.split('\n\n').str.len().sum()],
})
display(corpus_audit)

assert len(articles) == 15
assert articles['text'].str.contains('\n\n', regex=False).any()
assert articles['topic_group'].nunique() >= 5
assert articles['region_group'].nunique() >= 6


## Deterministic real-world holdout

Five articles are held out **before any TeAL fitting**. The set is deliberately topic-diverse rather than random: politics, sports, culture, and disasters are all represented. These texts will later be transformed only through reloaded frozen translators.

In [ ]:
HOLDOUT_IDS = [2, 7, 10, 14, 15]
train = articles.loc[~articles['article_id'].isin(HOLDOUT_IDS)].reset_index(drop=True)
held_out = articles.loc[articles['article_id'].isin(HOLDOUT_IDS)].reset_index(drop=True)

print('Training articles:', train['article_id'].tolist())
print('Held-out articles:', held_out['article_id'].tolist())
display(held_out[['article_id', 'title', 'topic_group', 'region_group']])

assert set(train['article_id']).isdisjoint(set(held_out['article_id']))
assert len(train) == 10 and len(held_out) == 5
assert held_out['topic_group'].nunique() >= 4


## Fit a representation inside TeAL

This pipeline intentionally mixes several translator types:

- `CountVectorizer`: learned text vocabulary;
- `FeatureTrimmer`: learned positional feature mask, represented lazily inside TeAL;
- `TfidfTransformer`: fitted matrix weighting;
- `SVD`: fitted dense dimensionality reduction.

Only the ten training articles enter this project.

In [ ]:
train_path = WORKDIR / 'train_articles.csv'
train[['article_id', 'title', 'topic_group', 'region_group', 'text']].to_csv(train_path, index=False)

project = teal.Project.create(PROJECT_PATH, name='real_world_standalone_acceptance')
source = project.read_csv(
    train_path,
    text_fields='text',
    metadata_fields=['article_id', 'title', 'topic_group', 'region_group'],
    batch_size=4,
)

count = CountVectorizer(
    text_field='text',
    lowercase=True,
    stop_words='english',
    min_df=1,
    ngram_range=(1, 2),
)
counts = project.translate(count, source, batch_size=3)['output']

trimmer = FeatureTrimmer(min_df=2, max_features=150)
trimmed = project.translate(trimmer, counts)['output']

tfidf = TfidfTransformer(norm='l2')
weighted = project.translate(tfidf, trimmed)['output']

svd = SVD(n_components=4, random_state=2025)
reduced = project.translate(svd, weighted)['output']

count_features = counts.get_data_columns()
trimmed_features = trimmed.get_data_columns()
count_matrix = counts.get_matrix()
trimmed_matrix = trimmed.get_matrix()

print('Count matrix:', count_matrix.shape, 'nnz=', count_matrix.nnz)
print('Features before trimming:', len(count_features))
print('Features after trimming:', len(trimmed_features))
print('SVD shape:', reduced.get_matrix().shape)

assert count_matrix.shape[0] == len(train)
assert 4 < len(trimmed_features) <= 150
assert reduced.get_matrix().shape == (len(train), 4)


In [ ]:
feature_sums = np.asarray(count_matrix.sum(axis=0)).ravel()
top_positions = np.argsort(feature_sums)[::-1][:30]
top_training_features = pd.DataFrame({
    'feature': [count_features[i] for i in top_positions],
    'count': feature_sums[top_positions].astype(int),
})
display(top_training_features)

print('First 50 retained features:')
print(trimmed_features[:50])

density_before = count_matrix.nnz / np.prod(count_matrix.shape)
density_after = trimmed_matrix.nnz / np.prod(trimmed_matrix.shape)
print(f'Count density before trimming: {density_before:.4f}')
print(f'Count density after trimming:  {density_after:.4f}')


## Kill the TeAL execution context, then use the translators normally

We save only IDs, close the project, reopen it to deserialize the frozen operators, and close it again **before** transforming held-out text. The actual held-out transformation below therefore uses ordinary Python/scientific-Python objects only.

In [ ]:
operator_ids = {
    'count': str(count.operator_id),
    'trimmer': str(trimmer.operator_id),
    'tfidf': str(tfidf.operator_id),
    'svd': str(svd.operator_id),
}
target_artifact_id = str(reduced.artifact_id)
project.close()

reopened = teal.Project.open(PROJECT_PATH)
frozen_count = reopened.get_operator(operator_ids['count'])
frozen_trimmer = reopened.get_operator(operator_ids['trimmer'])
frozen_tfidf = reopened.get_operator(operator_ids['tfidf'])
frozen_svd = reopened.get_operator(operator_ids['svd'])
reopened.close()

heldout_texts = held_out['text'].tolist()
X_count = frozen_count.translate(heldout_texts)
X_trimmed = frozen_trimmer.translate(X_count)
X_tfidf = frozen_tfidf.translate(X_trimmed)
Z_manual = frozen_svd.translate(X_tfidf)

print('Held-out count:', X_count.shape)
print('Held-out trimmed:', X_trimmed.shape)
print('Held-out TF-IDF:', X_tfidf.shape)
print('Held-out SVD:', Z_manual.shape)

assert X_count.shape[0] == len(held_out)
assert X_trimmed.shape[1] == len(trimmed_features)
assert Z_manual.shape == (len(held_out), 4)


## Compare manual standalone composition with frozen GeCo-style replay

This checks the architecture from two independent entry points. Replay is allowed to inspect persisted lineage, but the actual user-facing translator stages should resolve to the same numerical representation. We also verify that replay itself is read-only with respect to TeAL artifacts and operations.

In [ ]:
reopened = teal.Project.open(PROJECT_PATH)
before_artifacts = len(reopened.list_artifacts())
before_operations = len(reopened.list_operations())
Z_replay = _replay_texts_like(
    reopened,
    target_artifact_id,
    heldout_texts,
    query=False,
)
after_artifacts = len(reopened.list_artifacts())
after_operations = len(reopened.list_operations())
reopened.close()

Z_replay = Z_replay.toarray() if sparse.issparse(Z_replay) else np.asarray(Z_replay)
np.testing.assert_allclose(Z_manual, Z_replay, rtol=1e-12, atol=1e-12)
assert before_artifacts == after_artifacts
assert before_operations == after_operations

max_difference = float(np.max(np.abs(Z_manual - Z_replay)))
print('Maximum absolute manual-vs-replay difference:', max_difference)
print('Artifacts changed:', after_artifacts - before_artifacts)
print('Operations changed:', after_operations - before_operations)


## Inspect what the representation is actually doing

Numerical equality is not enough for a real-world sanity check. These cells expose the held-out coordinates and the strongest TF-IDF features so we can look for obvious scientific nonsense.

In [ ]:
coordinates = held_out[['article_id', 'title', 'topic_group']].copy()
for i in range(Z_manual.shape[1]):
    coordinates[f'svd_{i}'] = Z_manual[:, i]
display(coordinates)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(coordinates['svd_0'], coordinates['svd_1'])
for _, row in coordinates.iterrows():
    ax.annotate(str(int(row['article_id'])), (row['svd_0'], row['svd_1']))
ax.set_xlabel('SVD 0')
ax.set_ylabel('SVD 1')
ax.set_title('Held-out Wikinews articles in the frozen training representation')
plt.show()


In [ ]:
tfidf_dense = X_tfidf.toarray() if sparse.issparse(X_tfidf) else np.asarray(X_tfidf)
top_term_rows = []
for row_index, article in held_out.iterrows():
    weights = tfidf_dense[row_index]
    strongest = np.argsort(weights)[::-1][:8]
    terms = [trimmed_features[i] for i in strongest if weights[i] > 0]
    top_term_rows.append({
        'article_id': int(article['article_id']),
        'topic': article['topic_group'],
        'title': article['title'],
        'top_weighted_features': ', '.join(terms),
    })
display(pd.DataFrame(top_term_rows))


## Real cardinality-changing + structured standalone translation

Now leave matrix representations entirely. We decompose held-out articles into their real paragraphs, clean punctuation/whitespace, and calculate multiple length measures—all **without a Project**. This samples the structured/cardinality-changing side of the new contracts.

In [ ]:
decomposer = DelimiterDecomposer(
    delimiter='\n\n',
    new_key='paragraph_id',
    text_field='text',
    output_text_field='paragraph_text',
)
parts = decomposer.translate(held_out['text'])

paragraphs = pd.DataFrame({
    'source_position': parts['source_positions'],
    'paragraph_id': parts['segment_ids'],
    'paragraph_text': parts['texts'],
})
paragraphs['article_id'] = paragraphs['source_position'].map(
    dict(enumerate(held_out['article_id'].astype(int)))
)
paragraphs['topic_group'] = paragraphs['source_position'].map(
    dict(enumerate(held_out['topic_group']))
)

cleaner = RegexCleaner(
    text_field='paragraph_text',
    output_field='clean_text',
    rules=[
        RegexReplaceRule(r"[^\w\s'-]+", ' '),
        RegexReplaceRule(r'\s+', ' '),
    ],
    strip=True,
)
paragraphs['clean_text'] = cleaner.translate(paragraphs['paragraph_text'])

lengths = TextLength({'clean_text': ['words', 'characters', 'log_words']})
length_frame = lengths.translate(paragraphs[['clean_text']])
paragraphs = pd.concat([paragraphs.reset_index(drop=True), length_frame.reset_index(drop=True)], axis=1)

display(paragraphs[['article_id', 'paragraph_id', 'topic_group', 'clean_text', 'clean_text_words']])
display(paragraphs.groupby('article_id')['clean_text_words'].agg(['count', 'min', 'median', 'max']))

assert len(paragraphs) == sum(len(str(text).split('\n\n')) for text in heldout_texts)
assert (paragraphs['clean_text_words'] > 0).all()
assert paragraphs.groupby('source_position')['paragraph_id'].min().eq(0).all()


## Optional: real spaCy runtime on the held-out corpus

This cell does **not** download a model. If `en_core_web_sm` is already installed, it exercises a real multi-output NLP translator on the held-out articles and displays actual sentence/token records. Otherwise it reports that the optional check was skipped.

In [ ]:
try:
    import spacy
    has_model = spacy.util.is_package('en_core_web_sm')
except Exception:
    has_model = False

if has_model:
    spacy_translator = SpacyTranslator(model='en_core_web_sm')
    linguistic = spacy_translator.translate(held_out['text'].iloc[:2])
    print('Sentence rows:', len(linguistic['sentences']))
    print('Token rows:', len(linguistic['tokens']))
    display(linguistic['sentences'].head(10))
    display(linguistic['tokens'].head(25))
    assert len(linguistic['sentences']) > 0
    assert len(linguistic['tokens']) > len(linguistic['sentences'])
else:
    print('Skipped: en_core_web_sm is not installed. No model was downloaded automatically.')


## Final inspection checklist

Before treating this notebook as a successful real-world acceptance run, inspect the displayed outputs rather than relying only on assertions:

- Do the learned high-frequency features look like plausible content terms rather than parsing artifacts?
- Did feature trimming retain a useful mixture of terms rather than collapsing the representation?
- Do the held-out articles receive nondegenerate TF-IDF and SVD representations?
- Do the strongest held-out features roughly correspond to each article's subject matter?
- Does manual standalone composition agree numerically with frozen representation replay?
- Did standalone/replay use leave the TeAL catalog unchanged?
- Do paragraph boundaries, cleaned text, and word counts look sensible on actual prose?
- If spaCy is available, do sentence/token keys and offsets look plausible?

The assertions below summarize the hard checks; the qualitative displays above are the point of using a notebook.

In [ ]:
assert np.isfinite(Z_manual).all()
assert not np.allclose(Z_manual, 0)
assert max_difference <= 1e-12
assert before_artifacts == after_artifacts
assert before_operations == after_operations
assert paragraphs['article_id'].nunique() == len(held_out)
assert paragraphs['clean_text_words'].median() > 10

print('All automated notebook checks passed.')
print('Now inspect the qualitative tables/plot above before signing off on the merge.')
